### 1.Comparing computation cost for GRUs, LSTM and regular RNNs

In [31]:
import torch
import torch.nn as nn
import time
import torch.optim as optim
import collections
from torch.utils.data import DataLoader, Dataset
import math

In [2]:
#!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126 --break-system-packages

## Importing dataset

In [5]:
#!pip3 install datasets --break-system-packages

In [6]:
#!pip3 install torchtext --break-system-packages

In [6]:
from datasets import load_dataset

dataset = load_dataset("wmt14", "de-en")

train = dataset["train"]

README.md: 0.00B [00:00, ?B/s]

de-en/train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

de-en/train-00001-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

de-en/train-00002-of-00003.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

de-en/validation-00000-of-00001.parquet:   0%|          | 0.00/474k [00:00<?, ?B/s]

de-en/test-00000-of-00001.parquet:   0%|          | 0.00/509k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

## Tokenizer

In [9]:
class SimpleTokenizer:
    def __init__(self,min_freq=2):
        self.word2idx = {"<pad>":0,"<sos>":1,"<eos>":2,"<unk>":3}
        self.idx2word = {v:k for k,v in self.word2idx.items()}
        self.min_freq = min_freq

    def build_vocab(self,sentences):
        tokens = [word.lower() for sent in sentences for word in sent.split()]
        counts = collections.Counter(tokens)
        for word, freq in counts.items():
            if freq >= self.min_freq:
                if word not in self.word2idx:
                    idx = len(self.word2idx)
                    self.word2idx[word] = idx
                    self.idx2word[idx] = word
    def encode(self,text):
        tokens = text.lower().split()
        return [1] + [self.word2idx.get(w,3) for w in tokens] + [2]

    def decode(self,ids):
        return " ".join([self.idx2word.get(i,"<unk>") for i in ids])


In [10]:
dataset = load_dataset("wmt14", "de-en", split='train', streaming=True).take(1000)
data = list(dataset)

In [14]:
# Initialize Tokenizers
de_tokenizer = SimpleTokenizer()
en_tokenizer = SimpleTokenizer()

# Build Vocab
de_tokenizer.build_vocab([item['translation']['de'] for item in data])
en_tokenizer.build_vocab([item['translation']['en'] for item in data])

## The RNN Architecture (Seq2Seq)

In [26]:
class Encoder(nn.Module):
    def __init__(self,input_dim,emb_dim,hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim,emb_dim)
        self.rnn = nn.LSTM(emb_dim,hid_dim)
    def forward(self,src):
        embedded = self.embedding(src)
        outputs,(hidden,cell) = self.rnn(embedded)
        return hidden,cell

In [27]:

class Decoder(nn.Module):
    def __init__(self,output_dim,emb_dim,hid_dim):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim,emb_dim)
        self.rnn = nn.LSTM(emb_dim,hid_dim)
        self.fc_out = nn.Linear(hid_dim,output_dim)

    def forward(self,input,hidden,cell):
        input = input.unsqueeze(0) # [1, batch_size]   
        embedded = self.embedding(input)
        output,(hidden,cell)= self.rnn(embedded,(hidden,cell))
        prediction = self.fc_out(output.squeeze(0))
        return prediction,hidden,cell

In [28]:
class Seq2Seq(nn.Module):
    def __init__(self,encoder,decoder,device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self,src,trg,teacher_forcing_ratio=0.5):
        trg_len = trg.shape[0]
        batch_size = trg.shape[1]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len,batch_size,trg_vocab_size).to(self.device)
        hidden,cell = self.encoder(src)

        input = trg[0,:1]
        for t in range(1,trg_len):
            output,hidden,cell = self.decoder(input,hidden,cell)
            outputs[t] = output
            top1 = output.argmax(1)
            #decide
            input = trg[t] if torch.rand(1).item() < teacher_forcing_ratio else top1  

        return outputs

In [29]:
## Training Loop

In [32]:
# Constants
HID_DIM = 256
EMB_DIM = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Instantiate Model
enc = Encoder(len(de_tokenizer.word2idx), EMB_DIM, HID_DIM)
dec = Decoder(len(en_tokenizer.word2idx), EMB_DIM, HID_DIM)
model = Seq2Seq(enc, dec, DEVICE).to(DEVICE)

optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss(ignore_index=0) # Ignore <pad>

# Mock training step (1 epoch)
model.train()
for item in data:
    src = torch.LongTensor(de_tokenizer.encode(item['translation']['de'])).unsqueeze(1).to(DEVICE)
    trg = torch.LongTensor(en_tokenizer.encode(item['translation']['en'])).unsqueeze(1).to(DEVICE)
    
    optimizer.zero_grad()
    output = model(src, trg)
    
    # Reshape for loss: output[1:] removes <sos>, flatten for CrossEntropy
    output_dim = output.shape[-1]
    output = output[1:].view(-1, output_dim)
    trg = trg[1:].view(-1)
    
    loss = criterion(output, trg)
    loss.backward()
    optimizer.step()

print(f"Training complete. Last Loss: {loss.item():.4f}")

Training complete. Last Loss: 5.1457


### LSTM

In [33]:
class EncoderLSTM(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers)
        
    def forward(self, src):
        # src: [src_len, batch_size]
        embedded = self.embedding(src)
        # LSTM returns output, (hidden, cell)
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

class DecoderLSTM(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers=2):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        
    def forward(self, input, hidden, cell):
        # input: [batch_size]
        input = input.unsqueeze(0) # [1, batch_size]
        embedded = self.embedding(input)
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0))
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        trg_len, batch_size = trg.shape
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        
        # Encoder context
        hidden, cell = self.encoder(src)
        
        # First input to decoder is <sos>
        input = trg[0, :]
        
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[t] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1
            
        return outputs

In [34]:
# Hyperparameters
INPUT_DIM = len(de_tokenizer.word2idx)
OUTPUT_DIM = len(en_tokenizer.word2idx)
HID_DIM = 512
EMB_DIM = 256
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

enc = EncoderLSTM(INPUT_DIM, EMB_DIM, HID_DIM).to(DEVICE)
dec = DecoderLSTM(OUTPUT_DIM, EMB_DIM, HID_DIM).to(DEVICE)
model = Seq2Seq(enc, dec, DEVICE).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0) # Ignore <pad>

# Simple Training loop
model.train()
for epoch in range(5):
    epoch_loss = 0
    for i, item in enumerate(data):
        src = torch.LongTensor(de_tokenizer.encode(item['translation']['de'])).unsqueeze(1).to(DEVICE)
        trg = torch.LongTensor(en_tokenizer.encode(item['translation']['en'])).unsqueeze(1).to(DEVICE)

        optimizer.zero_grad()
        output = model(src, trg)
        
        # Cut off <sos> and reshape for loss
        output = output[1:].view(-1, OUTPUT_DIM)
        trg = trg[1:].view(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
        if i % 500 == 0:
            print(f"Step {i}, Loss: {loss.item():.4f}")

    print(f"Epoch {epoch} Avg Loss: {epoch_loss/len(data):.4f}")

Step 0, Loss: 7.5924
Step 500, Loss: 5.9775
Epoch 0 Avg Loss: 5.6021
Step 0, Loss: 3.8465
Step 500, Loss: 5.5172
Epoch 1 Avg Loss: 5.0982
Step 0, Loss: 3.8170
Step 500, Loss: 5.3458
Epoch 2 Avg Loss: 4.8473
Step 0, Loss: 3.3504
Step 500, Loss: 5.3082
Epoch 3 Avg Loss: 4.6421
Step 0, Loss: 3.1301
Step 500, Loss: 5.1637
Epoch 4 Avg Loss: 4.4604


## GRu

In [35]:
class EncoderGRU(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim, n_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers)
        
    def forward(self, src):
        # src: [src_len, batch_size]
        embedded = self.embedding(src)
        # GRU returns output and hidden (no cell state)
        outputs, hidden = self.rnn(embedded)
        return hidden

class DecoderGRU(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers=1):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim, n_layers)
        self.fc_out = nn.Linear(hid_dim, output_dim)
        
    def forward(self, input, hidden):
        # input: [batch_size]
        input = input.unsqueeze(0) # [1, batch_size]
        embedded = self.embedding(input)
        
        output, hidden = self.rnn(embedded, hidden)
        prediction = self.fc_out(output.squeeze(0))
        
        return prediction, hidden

class Seq2SeqGRU(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        trg_len, batch_size = trg.shape
        trg_vocab_size = self.decoder.output_dim
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        
        # Encoder context (Hidden state only)
        hidden = self.encoder(src)
        
        input = trg[0, :] # First token is <sos>
        
        for t in range(1, trg_len):
            output, hidden = self.decoder(input, hidden)
            outputs[t] = output
            
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[t] if teacher_force else top1
            
        return outputs

In [36]:
## Training Setup

In [37]:
# Assuming de_tokenizer and en_tokenizer are already built from previous steps
INPUT_DIM = len(de_tokenizer.word2idx)
OUTPUT_DIM = len(en_tokenizer.word2idx)
HID_DIM = 512
EMB_DIM = 256
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

enc = EncoderGRU(INPUT_DIM, EMB_DIM, HID_DIM).to(DEVICE)
dec = DecoderGRU(OUTPUT_DIM, EMB_DIM, HID_DIM).to(DEVICE)
model = Seq2SeqGRU(enc, dec, DEVICE).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0) # Ignore <pad>

# Training loop
model.train()
for i, item in enumerate(data):
    src = torch.LongTensor(de_tokenizer.encode(item['translation']['de'])).unsqueeze(1).to(DEVICE)
    trg = torch.LongTensor(en_tokenizer.encode(item['translation']['en'])).unsqueeze(1).to(DEVICE)

    optimizer.zero_grad()
    output = model(src, trg)
    
    # Loss calculation (skip <sos>)
    output_dim = output.shape[-1]
    output = output[1:].view(-1, output_dim)
    trg = trg[1:].view(-1)
    
    loss = criterion(output, trg)
    loss.backward()
    
    # Clip gradients to prevent exploding gradients (common in GRUs)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if i % 100 == 0:
        print(f"Iteration {i}, Loss: {loss.item():.4f}")

Iteration 0, Loss: 7.5736
Iteration 100, Loss: 5.8802
Iteration 200, Loss: 6.2209
Iteration 300, Loss: 5.8840
Iteration 400, Loss: 5.3351
Iteration 500, Loss: 6.0714
Iteration 600, Loss: 5.8337
Iteration 700, Loss: 4.8727
Iteration 800, Loss: 5.2224
Iteration 900, Loss: 5.4131
